# Smoke test Colab / Drive — setup và integrity

Chạy độc lập sau **Restart Kernel**, không cần notebook 03/04 hay biến cũ. Notebook không publish source, không nạp model, không gọi API, không train và không mở CSV test.

1. Chọn Colab kernel. Trong VS Code chạy **Colab: Mount Google Drive to Server...**, chạy cell extension tạo và hoàn tất xác thực. Trên Colab browser dùng chức năng mount được hỗ trợ. Kết nối kernel chưa mount Drive.
2. Source phải được publish đầy đủ trước khi chạy. Không có Drive desktop: làm đúng bước publish thủ công **một folder payload** trong [HANDOVER_GUIDE](../docs/HANDOVER_GUIDE.md#không-có-drive-for-desktop); không upload từng `.py`.
3. Điền `EXPECTED_RUNTIME_PAYLOAD` bằng SHA-256 in từ lệnh stage/publish local. Chỉ cần sửa `DRIVE_ROOT` nếu owner đã thay canonical layout. **Không lấy expected digest từ Drive.**

Run All chỉ setup/import/session/integrity. Nếu FAIL, sửa blocker đầu tiên rồi chạy lại từ setup; SKIPPED là bước phụ thuộc chưa chạy. Kết quả lưu trước đây không chứng minh runtime hiện tại đã PASS.

In [11]:
import json, platform, runpy
from pathlib import Path

# Chỉ sửa hai giá trị này; digest lấy từ Windows stage/publish, không lấy từ Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/ML_Projects/disease-prediction-ml')
EXPECTED_RUNTIME_PAYLOAD = ""
RUNTIME = ROOT = session = prepared = final_result = None
checks = []
step = "Kernel /content"
try:
    if platform.system() != 'Linux' or not Path('/content').is_dir():
        raise RuntimeError("Chọn Colab kernel; notebook này không dùng Python local.")
    checks.append(dict(check=step, status="PASS", detail="Linux /content; bootstrap sẽ xác minh Colab."))
    step = "Drive mount"
    if not Path('/content/drive/MyDrive').is_dir():
        raise FileNotFoundError("Drive chưa mount. Chạy Colab: Mount Google Drive to Server... và cell xác thực do extension tạo, rồi chạy lại setup.")
    checks.append(dict(check=step, status="PASS", detail="MyDrive truy cập được trên kernel hiện tại."))
    step = "Published source"
    entry = DRIVE_ROOT / 'src/data/colab_runtime.py'
    if not entry.is_file():
        raise FileNotFoundError(f"Drive đã mount nhưng thiếu {entry}. Kiểm tài khoản/root và publish toàn payload theo HANDOVER_GUIDE; notebook không tự publish.")
    checks.append(dict(check=step, status="PASS", detail=str(entry)))
    step = "Source release / bootstrap"
    RUNTIME = runpy.run_path(str(entry))['bootstrap_runtime'](
        DRIVE_ROOT, phase=3, expected_payload_sha256=EXPECTED_RUNTIME_PAYLOAD)
    ROOT = RUNTIME.paths.project_root
    checks.append(dict(check=step, status="PASS", detail=f"Windows expected = Drive = VM: {RUNTIME.payload_sha256}"))
except Exception as exc:
    checks.append(dict(check=step, status="FAIL", detail=f"{type(exc).__name__}: {exc}"))
    print(f"BLOCKER [{step}]: {type(exc).__name__}: {exc}")

BLOCKER [Source release / bootstrap]: TypeError: bootstrap_runtime() got an unexpected keyword argument 'expected_payload_sha256'


## Imports và cấu hình từ source đã kiểm checksum

Các script chỉ được nạp **định nghĩa** từ `ROOT/scripts/`; không thực thi audit, benchmark hoặc publisher. Không cài dependency tự động.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
dependencies_ready = False
if RUNTIME is None:
    checks.append(dict(check="Imports / configs / scripts", status="SKIPPED", detail="Sửa blocker setup trước."))
else:
    try:
        import importlib
        from src.data.colab_runtime import PUBLIC_CONFIGS, RUNTIME_SCRIPTS
        from src.models.notebook import ColabTraining
        for name in ("src.agent.conversation", "src.agent.extraction", "src.agent.provider", "src.models.final_evaluation"):
            module = importlib.import_module(name)
            if not Path(module.__file__).resolve().is_relative_to(ROOT.resolve()):
                raise RuntimeError(f"Module {name} từ source khác; Restart Kernel.")
        for name in sorted(PUBLIC_CONFIGS):
            json.loads((ROOT / "configs" / name).read_text(encoding="utf-8-sig"))
        for name in sorted(RUNTIME_SCRIPTS):
            runpy.run_path(str(ROOT / name))
        dependencies_ready = True
        checks.append(dict(check="Imports / configs / scripts", status="PASS", detail=f"8 configs; {len(RUNTIME_SCRIPTS)} script definitions; Phase 5B + final evaluation imports."))
    except Exception as exc:
        checks.append(dict(check="Imports / configs / scripts", status="FAIL", detail=f"{type(exc).__name__}: {exc}"))

## Session mới và integrity Phase 2 chỉ đọc

`ColabTraining.open` kiểm bảy file train/validation/metadata cùng schema/mapping, không parse CSV hoặc gọi `load_data`. Bootstrap dùng probe đọc/ghi tạm có sẵn; không ghi metadata, checkpoint hay manifest. Checkpoint và môi trường pickle được kiểm riêng trong preflight notebook 06; smoke này không chứng minh đã reload model.

In [14]:
if not dependencies_ready:
    checks.append(dict(check="Session / Phase 2 integrity", status="SKIPPED", detail="Chưa có source/imports hợp lệ."))
else:
    try:
        session = ColabTraining.open(RUNTIME)
        if session.data is not None or session.identity is not None:
            raise RuntimeError("Session phải mới, chưa load dữ liệu huấn luyện hoặc tạo experiment.")
        features, labels = session.inputs["schema"]["features"], session.inputs["labels"]["classes"]
        if len(features) != 375 or len(labels) != 587:
            raise ValueError("Schema phải có 375 features / 587 labels.")
        checks.append(dict(check="Session / Phase 2 integrity", status="PASS", detail=f"7 checksums; 375 features / 587 labels theo thứ tự đã khóa; manifest={session.inputs['manifest_sha256']}"))
    except Exception as exc:
        session = None
        checks.append(dict(check="Session / Phase 2 integrity", status="FAIL", detail=f"{type(exc).__name__}: {exc}"))

## Kết quả của lần chạy hiện tại

PASS chỉ áp dụng bước có bằng chứng trong kernel này. Checkpoint reload và Gemini live chưa được thử ở đây. Khi runtime chưa bootstrap được, Drive cloud sync vẫn NOT VERIFIED.

In [15]:
from IPython.display import Markdown, display

def status_table(rows):
    def clean(value):
        return str(value).replace("|", "/").replace("\n", " ")
    display(Markdown("| Kiểm tra | Trạng thái | Bằng chứng / hành động |\n|---|---|---|\n" +
                     "\n".join("| " + " | ".join(clean(r[k]) for k in ("check", "status", "detail")) + " |" for r in rows)))

status_table(checks)

smoke_result = {
    "status": "PASS" if session is not None else "FAIL",
    "checks": checks,
    "runtime_payload_sha256": RUNTIME.payload_sha256 if RUNTIME else None,
    "source_root": str(ROOT) if ROOT else None,
    "storage_root": str(DRIVE_ROOT),
    "session_initialized": session is not None,
    "checkpoint_reload": "NOT VERIFIED",
    "gemini_live": "NOT VERIFIED",
    "test_payload_opened": False,
    "models_trained": 0,
}
print(json.dumps(smoke_result, ensure_ascii=False, indent=2))

| Kiểm tra | Trạng thái | Bằng chứng / hành động |
|---|---|---|
| Kernel /content | PASS | Linux /content; bootstrap sẽ xác minh Colab. |
| Drive mount | PASS | MyDrive truy cập được trên kernel hiện tại. |
| Published source | PASS | /content/drive/MyDrive/ML_Projects/disease-prediction-ml/src/data/colab_runtime.py |
| Source release / bootstrap | FAIL | TypeError: bootstrap_runtime() got an unexpected keyword argument 'expected_payload_sha256' |
| Imports / configs / scripts | SKIPPED | Sửa blocker setup trước. |
| Session / Phase 2 integrity | SKIPPED | Chưa có source/imports hợp lệ. |

{
  "status": "FAIL",
  "checks": [
    {
      "check": "Kernel /content",
      "status": "PASS",
      "detail": "Linux /content; bootstrap sẽ xác minh Colab."
    },
    {
      "check": "Drive mount",
      "status": "PASS",
      "detail": "MyDrive truy cập được trên kernel hiện tại."
    },
    {
      "check": "Published source",
      "status": "PASS",
      "detail": "/content/drive/MyDrive/ML_Projects/disease-prediction-ml/src/data/colab_runtime.py"
    },
    {
      "check": "Source release / bootstrap",
      "status": "FAIL",
      "detail": "TypeError: bootstrap_runtime() got an unexpected keyword argument 'expected_payload_sha256'"
    },
    {
      "check": "Imports / configs / scripts",
      "status": "SKIPPED",
      "detail": "Sửa blocker setup trước."
    },
    {
      "check": "Session / Phase 2 integrity",
      "status": "SKIPPED",
      "detail": "Chưa có source/imports hợp lệ."
    }
  ],
  "runtime_payload_sha256": null,
  "source_root": null,
  "storage_

## Nếu còn blocker

- **Drive mount FAIL:** mount trên đúng server; không tạo thư mục Drive giả bằng `mkdir`.
- **Published source FAIL:** mount đã có nhưng sai account/root hoặc chưa publish. Làm bước publish riêng trong [HANDOVER_GUIDE](../docs/HANDOVER_GUIDE.md#không-có-drive-for-desktop), sau đó mở VM mới, mount lại và chạy notebook này; không thêm cell publish vào smoke.
- **Source release FAIL:** kiểm SHA-256 local, chờ đủ payload trên cloud rồi Restart Kernel; không sửa digest theo Drive hoặc bỏ integrity check.
- **Imports/session FAIL:** xem tên dependency/file và expected/actual checksum trong lỗi. Cài đúng requirements một cách chủ động nếu cần; không preprocess/train để bù artifact thiếu.
- **Checkpoint/môi trường:** notebook 06 có preflight chỉ đọc riêng. Không bật final-test gate để sửa lỗi.

Gửi JSON `smoke_result` và lỗi đầu tiên. STOP sau smoke; không Run All notebook 03/04 để thử môi trường.

In [17]:
import sys, sklearn, numpy, scipy, joblib
print("Python:", sys.version)
print("sklearn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
sklearn: 1.6.1
NumPy: 2.1.3
SciPy: 1.16.3
joblib: 1.6.0
